# CSI Prediction – Streamlined Inference (4 Models)

Loads trained checkpoints for all four models and evaluates on multiple test scenarios.

| # | Model | Type |
|---|-------|------|
| 1 | **GRU-Attn-DSLH** | Proposed (GRU + Gated Fusion Attention + DSLH) |
| 2 | **Vanilla-LSTM** | Baseline |
| 3 | **Vanilla-GRU** | Baseline |
| 4 | **LinFormer** | Baseline |

## Outputs
- Per-dataset NMSE (dB)
- Per-frame MSE across prediction horizon
- Time-series forecast vs ground truth
- Error histograms & heatmaps
- Cross-dataset comparison bar chart & leaderboard

In [ ]:
# ============================================================
# Cell 1: Setup & Imports
# ============================================================
import os, math, time, warnings, gc, copy, random, platform
from dataclasses import dataclass
from typing import Dict, List, Optional, Union
from bisect import bisect_right

import numpy as np
import h5py
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    if platform.system() == 'Windows':
        from torch.multiprocessing import set_start_method
        set_start_method('spawn', force=True)
except RuntimeError:
    pass

SEED = 43
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
print(f'Device: {DEVICE} | Seed: {SEED}')

In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================
@dataclass
class Cfg:
    # --- Paths (update to your local paths) ---
    out_dir: str = 'trained_streamlined'  # folder with best_*.pt checkpoints

    # --- Data settings (must match training) ---
    use_speed_feature: bool = True
    norm_per_run: bool = True
    norm_fit_train_only: bool = False
    safety_gap: bool = True
    npast: int = 128
    nfuture: int = 8
    train_frac: float = 0.8
    stride: int = 1

    # --- Model dims (must match training) ---
    d_model: int = 256
    d_model_linformer: int = 256
    d_model_rnn: int = 256
    n_layers: int = 3
    ffn_mult: int = 4
    dropout: float = 0.3

    # --- Proposed model flags (must match training) ---
    attn_scale: bool = True
    attn_dropout: float = 0.0
    query_proj: bool = False
    gru_residual: bool = False
    gated_fusion: bool = True
    stacked_gru: bool = False
    bidirectional: bool = False

    # --- Loader ---
    batch_size: int = 256
    num_workers: int = 0

cfg = Cfg()

# --- Models to evaluate ---
MODEL_TYPES = ['GRU-Attn-DSLH', 'Vanilla-LSTM', 'Vanilla-GRU', 'LinFormer']

# Optional: explicit checkpoint overrides (leave empty to auto-resolve)
CKPT_OVERRIDES: Dict[str, str] = {
    # 'GRU-Attn-DSLH': r'path\to\best_GRU-Attn-DSLH.pt',
}

# --- Test datasets ---
DATASETS: Dict[str, str] = {
    'Scenario I':   r'C:\Users\MOHUSSAIN25\Downloads\dataset_test_general.mat',
    'Scenario II':  r'C:\Users\MOHUSSAIN25\Downloads\dataset_test_pedestrian.mat',
    'Scenario III': r'C:\Users\MOHUSSAIN25\Downloads\dataset_test_umi_nlos.mat',
    'Scenario IV':  r'C:\Users\MOHUSSAIN25\Downloads\dataset_test_high_speed.mat',
    'Scenario V':   r'C:\Users\MOHUSSAIN25\Downloads\dataset_test_uma_los.mat',
}

print('Models:', MODEL_TYPES)
print('Datasets:')
for k, v in DATASETS.items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 3: Data Loading (HDF5 / MATLAB v7.3)
# ============================================================

def _parse_complex(arr):
    if arr.dtype.names is not None:
        if 'r' in arr.dtype.names and 'i' in arr.dtype.names:
            return arr['r'] + 1j * arr['i']
        elif 'real' in arr.dtype.names and 'imag' in arr.dtype.names:
            return arr['real'] + 1j * arr['imag']
    return arr


def _load_h5_cell_array(f, dataset_name):
    if dataset_name not in f:
        raise KeyError(f"Dataset '{dataset_name}' not found in HDF5 file.")
    ds = f[dataset_name]
    data_list = []
    refs = np.array(ds).flatten()
    for ref in refs:
        try:
            item = f[ref]
            vals = _parse_complex(np.array(item))
            data_list.append(vals)
        except Exception as e:
            warnings.warn(f"Could not dereference item in {dataset_name}: {e}")
    return data_list


def _to_H_time_major(ci_coeff):
    X = _parse_complex(ci_coeff)
    if X.ndim != 4:
        raise ValueError(f"Expected 4 dims, got {X.shape}")
    T, P, Tx, Rx = X.shape
    H = X.sum(axis=1).reshape(T, -1)
    return np.concatenate([H.real, H.imag], axis=1).astype(np.float32)


class QuaDRiGaRunWindows(Dataset):
    def __init__(self, seq_pairs, npast, nfuture, stride=1):
        self.npast, self.nfuture, self.stride = npast, nfuture, stride
        self.blocks, self.cum_counts = [], []
        self.feature_dim = self.target_dim = None
        total = 0
        for X, Y in seq_pairs:
            X = np.asarray(X, dtype=np.float32)
            Y = np.asarray(Y, dtype=np.float32)
            M = (X.shape[0] - (npast + nfuture)) // stride + 1
            if M <= 0: continue
            self.blocks.append({
                'X': torch.from_numpy(np.ascontiguousarray(X)),
                'Y': torch.from_numpy(np.ascontiguousarray(Y)),
            })
            total += M
            self.cum_counts.append(total)
            if self.feature_dim is None: self.feature_dim = X.shape[1]
            if self.target_dim is None:  self.target_dim  = Y.shape[1]
        self.total = total

    def __len__(self): return self.total

    def __getitem__(self, idx):
        block_idx = bisect_right(self.cum_counts, idx)
        prev = 0 if block_idx == 0 else self.cum_counts[block_idx - 1]
        start = (idx - prev) * self.stride
        blk = self.blocks[block_idx]
        return (blk['X'][start:start + self.npast],
                blk['Y'][start + self.npast:start + self.npast + self.nfuture])


def build_test_loader(mat_path: str, cfg: Cfg):
    """Build a test-only DataLoader from a single .mat file."""
    with h5py.File(mat_path, 'r') as f:
        csi_list  = _load_h5_cell_array(f, 'csi_dataset')
        speed_list = _load_h5_cell_array(f, 'speed_dataset')

    m = min(len(csi_list), len(speed_list))
    test_runs = []
    expected_in_dim = expected_out_dim = None

    for ci, spd in zip(csi_list[:m], speed_list[:m]):
        spd = np.asarray(spd).flatten().reshape(-1, 1).astype(np.float32)
        try:
            X_base = _to_H_time_major(ci)
        except Exception:
            continue
        if X_base.shape[0] <= 1: continue

        Cchan = X_base.shape[1]
        Y_delta = X_base[1:] - X_base[:-1]
        X_base  = X_base[:-1]
        spd = spd[:X_base.shape[0]]
        N_joint = min(X_base.shape[0], Y_delta.shape[0], spd.shape[0])
        if N_joint <= 0: continue
        X_base, Y_delta, spd = X_base[:N_joint], Y_delta[:N_joint], spd[:N_joint]

        X_feat = np.concatenate([X_base, spd], axis=1) if cfg.use_speed_feature else X_base

        # Use only the test partition
        N = X_feat.shape[0]
        cut = int(cfg.train_frac * N)
        gap = (cfg.npast + cfg.nfuture - 1) if cfg.safety_gap else 0

        if cfg.norm_per_run:
            stats_src = X_feat[:cut] if cfg.norm_fit_train_only else X_feat
            mu_ch = stats_src[:, :Cchan].mean(0, keepdims=True)
            sd_ch = stats_src[:, :Cchan].std(0, keepdims=True) + 1e-8
            Xn_ch = (X_feat[:, :Cchan] - mu_ch) / sd_ch
            if cfg.use_speed_feature:
                sp = X_feat[:, Cchan:]
                mu_sp = sp.mean(0, keepdims=True)
                sd_sp = sp.std(0, keepdims=True) + 1e-8
                Xn = np.concatenate([Xn_ch, (sp - mu_sp) / sd_sp], axis=1)
            else:
                Xn = Xn_ch
            Yn = (Y_delta - mu_ch) / sd_ch
        else:
            Xn, Yn = X_feat, Y_delta

        if expected_in_dim is None:
            expected_in_dim = Xn.shape[1]
            expected_out_dim = Yn.shape[1]
        elif Xn.shape[1] != expected_in_dim:
            continue

        Xte, Yte = Xn[cut + gap:], Yn[cut + gap:]
        if Xte.shape[0] >= cfg.npast + cfg.nfuture:
            test_runs.append((Xte, Yte))

    if not test_runs:
        raise RuntimeError(f'No valid test runs from {mat_path}')

    test_ds = QuaDRiGaRunWindows(test_runs, cfg.npast, cfg.nfuture, cfg.stride)
    test_dl = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=(DEVICE == 'cuda'))
    print(f'  Test windows: {len(test_ds)} | in={expected_in_dim} out={expected_out_dim}')
    return test_dl, expected_in_dim, expected_out_dim

In [ ]:
# ============================================================
# Cell 4: Model Architectures (all 4 models)
# ============================================================

# --- Shared Components ---

class DSLH(nn.Module):
    """Dimension-wise Separable Linear Head."""
    def __init__(self, Np, Nf, d_model, out_dim):
        super().__init__()
        self.W_time = nn.Parameter(torch.randn(Np, Nf) * (1.0 / math.sqrt(Np)))
        self.W_ch   = nn.Linear(d_model, out_dim, bias=True)

    def forward(self, F):
        G = torch.matmul(F.transpose(1, 2), self.W_time).transpose(1, 2)
        return self.W_ch(G)


class Attention(nn.Module):
    """Global Luong (dot) attention."""
    def __init__(self, hidden_dim, attn_dropout=0.0, scale=False):
        super().__init__()
        self.drop  = nn.Dropout(attn_dropout) if attn_dropout > 0 else nn.Identity()
        self.scale = 1.0 / math.sqrt(hidden_dim) if scale else 1.0

    def forward(self, enc_out, query):
        scores = torch.einsum('bth,bh->bt', enc_out, query) * self.scale
        w = self.drop(torch.softmax(scores, dim=-1))
        ctx = torch.einsum('bt,bth->bh', w, enc_out)
        return ctx, w


# --- LinFormer Components ---

class TMLP(nn.Module):
    def __init__(self, N, d_model, dropout=0.0):
        super().__init__()
        self.W1  = nn.Parameter(torch.randn(N, N) * (1.0 / math.sqrt(N)))
        self.W2  = nn.Parameter(torch.randn(N, N) * (1.0 / math.sqrt(N)))
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        xt = x.transpose(1, 2)
        return torch.matmul(self.drop(self.act(torch.matmul(xt, self.W1))), self.W2).transpose(1, 2)


class EncoderBlock(nn.Module):
    def __init__(self, N, d_model, ffn_mult=1, dropout=0.1):
        super().__init__()
        self.tmlp = TMLP(N, d_model, dropout)
        self.ln1  = nn.LayerNorm(d_model)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ffn_mult * d_model), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(ffn_mult * d_model, d_model), nn.Dropout(dropout),
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.ln1(x + self.tmlp(x))
        return self.ln2(x + self.ffn(x))


# ===================== MODEL 1: GRU-Attn-DSLH (Proposed) =====================

class GRUAttnBlock(nn.Module):
    def __init__(self, in_dim, hidden_dim, cfg, num_layers=1):
        super().__init__()
        self.bidirectional = cfg.bidirectional
        self.residual      = cfg.gru_residual
        self.gated_fusion  = cfg.gated_fusion

        self.rnn = nn.GRU(in_dim, hidden_dim, num_layers, batch_first=True,
                          dropout=cfg.dropout if num_layers > 1 else 0.0,
                          bidirectional=self.bidirectional)
        enc_dim = hidden_dim * (2 if self.bidirectional else 1)
        self.attn = Attention(enc_dim, attn_dropout=cfg.attn_dropout, scale=cfg.attn_scale)

        self.q_proj = (nn.Sequential(nn.Linear(enc_dim, enc_dim), nn.LayerNorm(enc_dim))
                       if cfg.query_proj else nn.Identity())

        fuse_dim = enc_dim * 2
        if self.gated_fusion:
            self.gate_net = nn.Sequential(
                nn.Linear(fuse_dim, enc_dim // 4), nn.ReLU(),
                nn.Linear(enc_dim // 4, enc_dim),
            )
        else:
            self.cat_proj = nn.Linear(fuse_dim, enc_dim)

        self.ln_post = nn.LayerNorm(enc_dim)
        self.drop    = nn.Dropout(cfg.dropout)
        self.res_proj = (nn.Linear(in_dim, enc_dim)
                         if self.residual and in_dim != enc_dim else nn.Identity())

    def forward(self, x):
        out, state = self.rnn(x)
        raw_q = (torch.cat([state[-2], state[-1]], -1) if self.bidirectional else state[-1])
        query = self.q_proj(raw_q)
        ctx, _ = self.attn(out, query)

        B, T, H = out.shape
        ctx_exp  = ctx.unsqueeze(1).expand(B, T, H)
        combined = torch.cat([out, ctx_exp], -1)

        if self.gated_fusion:
            z = torch.sigmoid(self.gate_net(combined))
            fused = z * out + (1 - z) * ctx_exp
        else:
            fused = self.cat_proj(combined)

        fused = self.drop(self.ln_post(fused))
        if self.residual:
            fused = fused + self.res_proj(x)
        return fused


class GRUAttnSeq(nn.Module):
    """Proposed: GRU + Gated Fusion Attention + DSLH."""
    def __init__(self, in_dim, out_dim, Np, Nf, cfg):
        super().__init__()
        dm = getattr(cfg, 'd_model_rnn', cfg.d_model)
        self.embed = nn.Linear(in_dim, dm)
        self.blocks = nn.ModuleList()
        if cfg.stacked_gru:
            for _ in range(cfg.n_layers):
                self.blocks.append(GRUAttnBlock(dm, dm, cfg, 1))
        else:
            self.blocks.append(GRUAttnBlock(dm, dm, cfg, cfg.n_layers))
        final_dim = dm * (2 if cfg.bidirectional else 1)
        self.head = DSLH(Np, Nf, final_dim, out_dim)

    def forward(self, x):
        h = self.embed(x)
        for blk in self.blocks: h = blk(h)
        return self.head(h)


# ===================== MODEL 2: Vanilla LSTM =====================

class VanillaLSTM(nn.Module):
    """Standard multi-layer LSTM + LayerNorm + DSLH head."""
    def __init__(self, in_dim, out_dim, Np, Nf, d_model=256, n_layers=3, dropout=0.3):
        super().__init__()
        self.embed = nn.Linear(in_dim, d_model)
        self.lstm  = nn.LSTM(d_model, d_model, n_layers, batch_first=True,
                             dropout=dropout if n_layers > 1 else 0.0)
        self.ln   = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.head = DSLH(Np, Nf, d_model, out_dim)

    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.head(self.drop(self.ln(out)))


# ===================== MODEL 3: Vanilla GRU =====================

class VanillaGRU(nn.Module):
    """Standard multi-layer GRU + LayerNorm + DSLH head."""
    def __init__(self, in_dim, out_dim, Np, Nf, d_model=256, n_layers=3, dropout=0.3):
        super().__init__()
        self.embed = nn.Linear(in_dim, d_model)
        self.gru   = nn.GRU(d_model, d_model, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.ln   = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.head = DSLH(Np, Nf, d_model, out_dim)

    def forward(self, x):
        out, _ = self.gru(self.embed(x))
        return self.head(self.drop(self.ln(out)))


# ===================== MODEL 4: LinFormer =====================

class LinFormer(nn.Module):
    """LinFormer-style encoder with TMLP blocks + DSLH head."""
    def __init__(self, in_dim, out_dim, Np, Nf, d_model=256, n_layers=3, ffn_mult=4, dropout=0.3):
        super().__init__()
        self.embed  = nn.Linear(in_dim, d_model)
        self.blocks = nn.ModuleList(
            [EncoderBlock(Np, d_model, ffn_mult, dropout) for _ in range(n_layers)])
        self.head = DSLH(Np, Nf, d_model, out_dim)

    def forward(self, x):
        h = self.embed(x)
        for blk in self.blocks: h = blk(h)
        return self.head(h)


print('All 4 model architectures defined.')

In [ ]:
# ============================================================
# Cell 5: Model Builder & Checkpoint Resolver
# ============================================================

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_model(model_type: str, in_dim: int, out_dim: int, cfg: Cfg) -> nn.Module:
    rnn_w = cfg.d_model_rnn
    lin_w = cfg.d_model_linformer

    if model_type == 'GRU-Attn-DSLH':
        return GRUAttnSeq(in_dim, out_dim, cfg.npast, cfg.nfuture, cfg)
    elif model_type == 'Vanilla-LSTM':
        return VanillaLSTM(in_dim, out_dim, cfg.npast, cfg.nfuture,
                           d_model=rnn_w, n_layers=cfg.n_layers, dropout=cfg.dropout)
    elif model_type == 'Vanilla-GRU':
        return VanillaGRU(in_dim, out_dim, cfg.npast, cfg.nfuture,
                          d_model=rnn_w, n_layers=cfg.n_layers, dropout=cfg.dropout)
    elif model_type == 'LinFormer':
        return LinFormer(in_dim, out_dim, cfg.npast, cfg.nfuture,
                         d_model=lin_w, n_layers=cfg.n_layers,
                         ffn_mult=cfg.ffn_mult, dropout=cfg.dropout)
    else:
        raise ValueError(f"Unknown model type: '{model_type}'")


def resolve_ckpt(model_type: str, cfg: Cfg) -> str:
    """Find the checkpoint file for a given model type."""
    # 1. Check explicit override
    if model_type in CKPT_OVERRIDES:
        p = CKPT_OVERRIDES[model_type]
        if os.path.isfile(p): return p

    # 2. Try standard naming patterns
    candidates = [
        os.path.join(cfg.out_dir, f'best_{model_type}.pt'),
        os.path.join(cfg.out_dir, f'best_{model_type}_final.pt'),
    ]
    # Also search for seed-based checkpoints
    import glob
    candidates += sorted(glob.glob(os.path.join(cfg.out_dir, f'best_{model_type}_seed*.pt')))

    for c in candidates:
        if os.path.isfile(c):
            return c

    raise FileNotFoundError(
        f"No checkpoint found for '{model_type}' in '{cfg.out_dir}'.\n"
        f"  Tried: {candidates}\n"
        f"  Tip: set CKPT_OVERRIDES['{model_type}'] = r'path/to/file.pt'"
    )


# Quick check
print('Checkpoint search paths:')
for mt in MODEL_TYPES:
    try:
        p = resolve_ckpt(mt, cfg)
        print(f'  {mt}: {p}')
    except FileNotFoundError as e:
        print(f'  {mt}: [NOT FOUND] {e}')

In [ ]:
# ============================================================
# Cell 6: Loss, Metrics & Visualization Helpers
# ============================================================

class WMSELoss(nn.Module):
    def __init__(self, Nf):
        super().__init__()
        self.register_buffer('w', torch.arange(1, Nf + 1, dtype=torch.float32) ** -0.5)

    def forward(self, pred, target):
        return (self.w.view(1, -1, 1) * (pred - target) ** 2).mean()


def mse_per_frame(pred, target):
    return ((pred - target) ** 2).mean(dim=(0, 2)).detach().cpu().numpy()


@torch.no_grad()
def evaluate(model, loader, Nf):
    model.eval()
    loss_fn = WMSELoss(Nf).to(DEVICE)
    tot_loss = sse = sst = total = 0
    pf_acc = None
    y_true_sample = y_pred_sample = None

    for X, Y in tqdm(loader, desc='  [eval]', leave=False):
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        B = X.size(0)
        P = model(X)
        tot_loss += loss_fn(P, Y).item() * B
        sse   += (P - Y).pow(2).sum().item()
        sst   += Y.pow(2).sum().item()
        total += B

        pf = mse_per_frame(P, Y)
        pf_acc = pf * B if pf_acc is None else pf_acc + pf * B

        if y_true_sample is None:
            y_true_sample = Y[0].cpu().numpy()
            y_pred_sample = P[0].cpu().numpy()

    nmse_db = 10.0 * math.log10(sse / (sst + 1e-12) + 1e-30)
    return {
        'loss':     tot_loss / max(total, 1),
        'nmse_db':  nmse_db,
        'per_frame_mse': pf_acc / total,
        'y_true':   y_true_sample,
        'y_pred':   y_pred_sample,
    }


# --- Plotting ---

def plot_forecast(ds, model, y_true, y_pred):
    t = np.arange(1, y_true.shape[0] + 1)
    plt.figure(figsize=(8, 3.5))
    plt.plot(t, y_true.mean(1), '-o', ms=4, label='Ground Truth')
    plt.plot(t, y_pred.mean(1), '--x', ms=5, label='Prediction')
    plt.title(f'{ds} | {model}: Forecast (avg channels)')
    plt.xlabel('Future step'); plt.ylabel('Value')
    plt.legend(); plt.grid(ls=':', alpha=.6); plt.tight_layout(); plt.show()


def plot_per_frame(ds, model, pf):
    t = np.arange(1, len(pf) + 1)
    plt.figure(figsize=(7, 3))
    plt.plot(t, pf, '-s', ms=4)
    plt.title(f'{ds} | {model}: Per-frame MSE')
    plt.xlabel('Future step'); plt.ylabel('MSE')
    plt.grid(ls=':', alpha=.6); plt.tight_layout(); plt.show()


def plot_error_hist(ds, model, y_true, y_pred):
    err = (y_pred - y_true).ravel()
    plt.figure(figsize=(7, 3))
    plt.hist(err, bins=40, alpha=.8, edgecolor='k')
    plt.title(f'{ds} | {model}: Error histogram')
    plt.xlabel('Error'); plt.ylabel('Count')
    plt.grid(ls=':', alpha=.6); plt.tight_layout(); plt.show()


def plot_error_heatmap(ds, model, y_true, y_pred, max_ch=64):
    err = np.abs(y_pred - y_true)[:, :max_ch]
    plt.figure(figsize=(8, 3.5))
    plt.imshow(err.T, aspect='auto', origin='lower')
    plt.colorbar(label='|error|')
    plt.title(f'{ds} | {model}: Error heatmap')
    plt.xlabel('Future step'); plt.ylabel('Channel')
    plt.tight_layout(); plt.show()


def plot_all(ds, model, metrics):
    print(f'\n  {model} on {ds}  |  WMSE: {metrics["loss"]:.6f}  |  NMSE: {metrics["nmse_db"]:.2f} dB')
    plot_forecast(ds, model, metrics['y_true'], metrics['y_pred'])
    plot_per_frame(ds, model, metrics['per_frame_mse'])
    plot_error_hist(ds, model, metrics['y_true'], metrics['y_pred'])
    plot_error_heatmap(ds, model, metrics['y_true'], metrics['y_pred'])


print('Evaluation & visualisation helpers ready.')

In [ ]:
# ============================================================
# Cell 7: Run Inference on All Datasets & Models
# ============================================================

all_results = []        # list of dicts
global_in_dim = None
global_out_dim = None

for ds_name, mat_path in DATASETS.items():
    print('\n' + '=' * 70)
    print(f'Dataset: {ds_name}')
    print(f'File:    {mat_path}')

    try:
        test_dl, IN_DIM, OUT_DIM = build_test_loader(mat_path, cfg)
    except Exception as e:
        print(f'  [SKIP] {e}')
        continue

    if global_in_dim is None:
        global_in_dim, global_out_dim = IN_DIM, OUT_DIM

    for model_type in MODEL_TYPES:
        print(f'\n  -- {model_type}')

        # Resolve checkpoint
        try:
            ckpt = resolve_ckpt(model_type, cfg)
        except FileNotFoundError as e:
            print(f'     [SKIP] {e}')
            continue

        # Build & load
        model = build_model(model_type, IN_DIM, OUT_DIM, cfg).to(DEVICE)
        print(f'     Params: {count_params(model)/1e6:.3f} M  |  Checkpoint: {os.path.basename(ckpt)}')

        try:
            state = torch.load(ckpt, map_location=DEVICE, weights_only=True)
            model.load_state_dict(state)
        except Exception as e:
            print(f'     [ERROR loading] {e}')
            del model; continue

        # Evaluate
        metrics = evaluate(model, test_dl, cfg.nfuture)
        metrics['dataset'] = ds_name
        metrics['model']   = model_type
        all_results.append(metrics)

        # Visualise
        plot_all(ds_name, model_type, metrics)

        del model
        gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    del test_dl
    gc.collect()

print('\n' + '=' * 70)
print('Inference complete.')

In [ ]:
# ============================================================
# Cell 8: Cross-Dataset Comparison & Leaderboard
# ============================================================
import pandas as pd

if not all_results:
    print('No results to compare.')
else:
    dataset_names = list(dict.fromkeys(r['dataset'] for r in all_results))  # preserve order
    model_names   = list(dict.fromkeys(r['model']   for r in all_results))

    # --- Leaderboard Table ---
    rows = []
    for m in model_names:
        entries = [r for r in all_results if r['model'] == m]
        row = {'Model': m, 'Avg NMSE (dB)': np.mean([e['nmse_db'] for e in entries])}
        for ds in dataset_names:
            e = next((r for r in entries if r['dataset'] == ds), None)
            row[ds] = e['nmse_db'] if e else float('nan')
        rows.append(row)

    df = pd.DataFrame(rows).sort_values('Avg NMSE (dB)')
    print('\n=== Leaderboard ===')
    print(df.to_string(index=False, float_format='{:.2f}'.format))

    # --- Bar Chart: NMSE per dataset ---
    hatches = ['/', '\\', '|', '-', '+', 'x']
    x = np.arange(len(dataset_names))
    width = 0.8 / max(1, len(model_names))

    fig, ax = plt.subplots(figsize=(max(8, len(dataset_names) * 1.5), 4.5))
    for i, m in enumerate(model_names):
        vals = []
        for ds in dataset_names:
            e = next((r for r in all_results if r['dataset'] == ds and r['model'] == m), None)
            vals.append(e['nmse_db'] if e else np.nan)
        ax.bar(x + i * width, vals, width=width, label=m,
               edgecolor='black', hatch=hatches[i % len(hatches)])

    ax.set_xticks(x + width * (len(model_names) - 1) / 2)
    ax.set_xticklabels(dataset_names, rotation=25, ha='right', fontsize=11)
    ax.set_ylabel('NMSE (dB)', fontsize=12)
    ax.set_title('NMSE per Dataset per Model', fontsize=13)
    ax.legend(fontsize=10); ax.grid(axis='y', ls=':', alpha=.5)
    plt.tight_layout(); plt.show()

    # --- Per-frame MSE averaged across datasets ---
    markers   = ['o', 's', '^', 'D']
    linestyles = ['-', '--', '-.', ':']
    t = np.arange(1, cfg.nfuture + 1)

    fig, ax = plt.subplots(figsize=(8, 4))
    for i, m in enumerate(model_names):
        entries = [r for r in all_results if r['model'] == m]
        avg_pf  = np.mean([e['per_frame_mse'] for e in entries], axis=0)
        pf_db   = 10 * np.log10(avg_pf + 1e-12)
        ax.plot(t, pf_db, marker=markers[i % 4], ls=linestyles[i % 4],
                ms=5, lw=1.5, label=m)

    ax.set_xlabel('Future Time Step', fontsize=12)
    ax.set_ylabel('MSE (dB)', fontsize=12)
    ax.set_title('Average Per-frame MSE across Datasets', fontsize=13)
    ax.legend(fontsize=10); ax.grid(True, ls=':', alpha=.6)
    plt.tight_layout(); plt.show()

    print('\nDone.')

In [ ]:
# ============================================================
# Cell 9: Efficiency Comparison (Params, FLOPs, Throughput)
# ============================================================

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print('Note: install `thop` for FLOP counting (pip install thop)')

if global_in_dim is None:
    print('Run inference first to determine input dimensions.')
else:
    eff_rows = []
    for mt in MODEL_TYPES:
        model = build_model(mt, global_in_dim, global_out_dim, cfg).to(DEVICE).eval()
        params_m = count_params(model) / 1e6

        # FLOPs
        gflops = 0.0
        if HAS_THOP:
            try:
                dummy = torch.randn(1, cfg.npast, global_in_dim).to(DEVICE)
                macs, _ = thop_profile(model, inputs=(dummy,), verbose=False)
                gflops = macs * 2 / 1e9
            except Exception:
                pass

        # Throughput
        dummy_bs = torch.randn(cfg.batch_size, cfg.npast, global_in_dim).to(DEVICE)
        with torch.no_grad():
            for _ in range(5): model(dummy_bs)  # warmup
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        n_runs = 30
        with torch.no_grad():
            for _ in range(n_runs): model(dummy_bs)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        throughput = (n_runs * cfg.batch_size) / (time.time() - t0)

        eff_rows.append({
            'Model': mt,
            'Params (M)': f'{params_m:.3f}',
            'GFLOPs': f'{gflops:.2f}' if gflops > 0 else 'N/A',
            'Throughput (samples/s)': f'{throughput:,.0f}',
        })
        del model

    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

    df_eff = pd.DataFrame(eff_rows)
    print('\n=== Efficiency Comparison ===')
    print(df_eff.to_string(index=False))